# EDA Workflow CI/CD Repositories

- analizzare qualità e copertura degli output estratti;
- confrontare report CSV e file realmente presenti;
- creare un dataset pulito a granularità file workflow;
- esportare artefatti finali pronti per analisi successive.

In [7]:
import csv
import json
import os
import re
import time
import warnings
from collections import Counter
from pathlib import Path

warnings.filterwarnings("ignore")

try:
    import pandas as pd
except Exception:
    pd = None

print("Python OK")
print(f"Pandas disponibile: {pd is not None}")

Python OK
Pandas disponibile: False


## 1) Configurazione notebook e dipendenze
Questo notebook usa solo librerie standard e, se presente, pandas per tabelle più comode.

## 2) Definizione parametri di esecuzione
Modifica solo questa cella per cambiare input/output, soglie e comportamento.

In [8]:
ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "output"
WORKFLOWS_DIR = OUTPUT_DIR / "workflows"
SUMMARY_JSON = OUTPUT_DIR / "extraction_summary.json"
REPORT_CSV = OUTPUT_DIR / "workflows_extraction_report.csv"
REPOS_CSV = OUTPUT_DIR / "repos_from_awesome_ai_agents.csv"

# Parametri modificabili
SAMPLE_FILE_COUNT = 5
YAML_SIGNATURE_KEYS = ("name:", "on:", "jobs:", "runs-on:")
MIN_VALID_WORKFLOW_BYTES = 30
DEBUG = False
SEED = 42

EXPORT_CSV = OUTPUT_DIR / "dataset_workflow_files_clean.csv"
EXPORT_PARQUET = OUTPUT_DIR / "dataset_workflow_files_clean.parquet"

print(f"ROOT: {ROOT}")
for p in [OUTPUT_DIR, WORKFLOWS_DIR, SUMMARY_JSON, REPORT_CSV, REPOS_CSV]:
    print(f"- {p} -> {'OK' if p.exists() else 'MISSING'}")

ROOT: /Users/giuseppepiosorrentino/awesome-ai-agents-cicd-extract
- /Users/giuseppepiosorrentino/awesome-ai-agents-cicd-extract/output -> OK
- /Users/giuseppepiosorrentino/awesome-ai-agents-cicd-extract/output/workflows -> OK
- /Users/giuseppepiosorrentino/awesome-ai-agents-cicd-extract/output/extraction_summary.json -> OK
- /Users/giuseppepiosorrentino/awesome-ai-agents-cicd-extract/output/workflows_extraction_report.csv -> OK
- /Users/giuseppepiosorrentino/awesome-ai-agents-cicd-extract/output/repos_from_awesome_ai_agents.csv -> OK


## 3) Refactor del codice in funzioni riutilizzabili
Le funzioni sotto separano caricamento, analisi e preparazione dataset.

In [9]:
def load_inputs(summary_path: Path, report_path: Path, repos_path: Path):
    summary = {}
    if summary_path.exists():
        summary = json.loads(summary_path.read_text(encoding="utf-8"))

    report_rows = []
    if report_path.exists():
        with report_path.open("r", encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)
            report_rows = list(reader)

    repos_rows = []
    if repos_path.exists():
        with repos_path.open("r", encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)
            repos_rows = list(reader)

    return summary, report_rows, repos_rows


def repo_to_dir(repo: str) -> str:
    return repo.replace("/", "__")


def infer_yaml_like(text: str, keys=YAML_SIGNATURE_KEYS) -> bool:
    t = text.lower()
    found = sum(1 for k in keys if k in t)
    return found >= 2


def list_workflow_files(workflows_dir: Path):
    files = []
    if not workflows_dir.exists():
        return files

    for p in workflows_dir.rglob("*"):
        if p.is_file():
            rel = p.relative_to(workflows_dir)
            parts = rel.parts
            repo_dir = parts[0] if parts else ""
            ext = p.suffix.lower() if p.suffix else ""
            files.append(
                {
                    "repo_dir": repo_dir,
                    "relative_path": str(rel).replace("\\", "/"),
                    "filename": p.name,
                    "extension": ext,
                    "size_bytes": p.stat().st_size,
                    "abs_path": str(p),
                }
            )
    return files


def map_report_status(report_rows):
    status_map = {}
    for r in report_rows:
        repo = (r.get("repo") or "").strip()
        status = (r.get("status") or "").strip()
        if repo:
            status_map[repo] = status
    return status_map


def build_clean_dataset(workflow_files, report_rows):
    report_status = map_report_status(report_rows)
    rows = []

    for wf in workflow_files:
        repo_dir = wf["repo_dir"]
        repo = repo_dir.replace("__", "/") if repo_dir else ""

        sample_text = ""
        try:
            with open(wf["abs_path"], "r", encoding="utf-8", errors="ignore") as f:
                sample_text = f.read(4000)
        except Exception:
            sample_text = ""

        rows.append(
            {
                "repo": repo,
                "repo_dir": repo_dir,
                "relative_path": wf["relative_path"],
                "filename": wf["filename"],
                "extension": wf["extension"] if wf["extension"] else "[no_ext]",
                "size_bytes": wf["size_bytes"],
                "inferred_is_yaml_like": infer_yaml_like(sample_text),
                "status_from_report": report_status.get(repo, ""),
            }
        )

    return rows


def compute_quality_metrics(summary, report_rows, repos_rows, workflow_files):
    report_repos = {(r.get("repo") or "").strip() for r in report_rows if (r.get("repo") or "").strip()}
    list_repos = {(r.get("repo") or "").strip() for r in repos_rows if (r.get("repo") or "").strip()}
    workflow_repo_dirs = {wf["repo_dir"] for wf in workflow_files if wf.get("repo_dir")}
    workflow_repos = {d.replace("__", "/") for d in workflow_repo_dirs}

    status_counter = Counter((r.get("status") or "").strip() or "[empty]" for r in report_rows)
    ext_counter = Counter(wf["extension"] if wf["extension"] else "[no_ext]" for wf in workflow_files)

    metrics = {
        "summary_totals": summary.get("totals", {}),
        "repos_in_repo_list": len(list_repos),
        "repos_in_report": len(report_repos),
        "repos_with_actual_files": len(workflow_repos),
        "workflow_file_count": len(workflow_files),
        "status_distribution": dict(status_counter),
        "extension_distribution": dict(ext_counter),
        "repos_with_files_not_in_report": sorted(workflow_repos - report_repos)[:20],
        "repos_in_report_without_files": sorted(report_repos - workflow_repos)[:20],
    }
    return metrics


def to_dataframe(rows):
    if pd is None:
        return None
    return pd.DataFrame(rows)


def save_dataset(rows, csv_path: Path, parquet_path: Path):
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    # Salvataggio CSV sempre disponibile
    if rows:
        fieldnames = list(rows[0].keys())
        with csv_path.open("w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
    else:
        csv_path.write_text("", encoding="utf-8")

    parquet_saved = False
    parquet_error = ""

    if pd is not None and rows:
        try:
            df = pd.DataFrame(rows)
            df.to_parquet(parquet_path, index=False)
            parquet_saved = True
        except Exception as e:
            parquet_error = str(e)

    return {
        "csv_path": str(csv_path),
        "parquet_path": str(parquet_path),
        "parquet_saved": parquet_saved,
        "parquet_error": parquet_error,
    }

## 4) Esecuzione step-by-step in celle
Questa sezione esegue i blocchi principali e mostra output intermedi utili al debug.

In [10]:
t0 = time.time()

summary, report_rows, repos_rows = load_inputs(SUMMARY_JSON, REPORT_CSV, REPOS_CSV)
workflow_files = list_workflow_files(WORKFLOWS_DIR)
metrics = compute_quality_metrics(summary, report_rows, repos_rows, workflow_files)
clean_rows = build_clean_dataset(workflow_files, report_rows)
clean_df = to_dataframe(clean_rows)

elapsed = time.time() - t0
print(f"Caricamento+analisi completati in {elapsed:.2f}s")
print("\nMetriche principali")
for k, v in metrics.items():
    if isinstance(v, dict):
        print(f"- {k}: {len(v)} voci")
    elif isinstance(v, list):
        print(f"- {k}: {len(v)} elementi (preview: {v[:5]})")
    else:
        print(f"- {k}: {v}")

print(f"\nRighe dataset pulito: {len(clean_rows)}")
if clean_df is not None:
    display(clean_df.head(10))
else:
    print("Pandas non disponibile: preview via lista")
    print(clean_rows[:3])

Caricamento+analisi completati in 0.73s

Metriche principali
- summary_totals: 5 voci
- repos_in_repo_list: 120
- repos_in_report: 120
- repos_with_actual_files: 79
- workflow_file_count: 5885
- status_distribution: 2 voci
- extension_distribution: 1 voci
- repos_with_files_not_in_report: 0 elementi (preview: [])
- repos_in_report_without_files: 20 elementi (preview: ['0xpayne/gpt-migrate', 'AutoPackAI/beebot', 'CR-Gjx/Suspicion-Agent', 'DataBassGit/AgentForge', 'LehengTHU/Agent4Rec'])

Righe dataset pulito: 5885
Pandas non disponibile: preview via lista
[{'repo': 'samholt/l2mac', 'repo_dir': 'samholt__l2mac', 'relative_path': 'samholt__l2mac/d0cd95b60c061f585870854d7c2e654497c7a30f23137fa2f5990e2bbdeed518', 'filename': 'd0cd95b60c061f585870854d7c2e654497c7a30f23137fa2f5990e2bbdeed518', 'extension': '[no_ext]', 'size_bytes': 2083, 'inferred_is_yaml_like': True, 'status_from_report': 'no_workflow_yaml'}, {'repo': 'calcom/cal.com', 'repo_dir': 'calcom__cal.com', 'relative_path': 'calcom_

In [11]:
print("Distribuzione status (report CSV):")
status_dist = Counter((r.get("status") or "").strip() or "[empty]" for r in report_rows)
for k, v in status_dist.most_common():
    print(f"- {k}: {v}")

print("\nDistribuzione estensioni (file reali):")
ext_dist = Counter(wf["extension"] if wf["extension"] else "[no_ext]" for wf in workflow_files)
for k, v in ext_dist.most_common(10):
    print(f"- {k}: {v}")

print("\nCampione file workflow (prime righe):")
for wf in workflow_files[:SAMPLE_FILE_COUNT]:
    p = Path(wf["abs_path"])
    print(f"\n### {wf['relative_path']} ({wf['size_bytes']} bytes)")
    try:
        with p.open("r", encoding="utf-8", errors="ignore") as f:
            lines = [next(f).rstrip("\n") for _ in range(8)]
    except StopIteration:
        pass
    except Exception as e:
        print(f"Errore lettura: {e}")
        continue
    else:
        for line in lines:
            print(line)

Distribuzione status (report CSV):
- no_workflow_yaml: 118
- gigawork_failed: 2

Distribuzione estensioni (file reali):
- [no_ext]: 5885

Campione file workflow (prime righe):

### samholt__l2mac/d0cd95b60c061f585870854d7c2e654497c7a30f23137fa2f5990e2bbdeed518 (2083 bytes)
# Sample workflow for building and deploying a VitePress site to GitHub Pages
#
name: Deploy VitePress site to Pages

on:
  # Runs on pushes targeting the `master` branch. Change this to `master` if you're
  # using the `master` branch as the default branch.
  push:

### calcom__cal.com/a74792f4af7900a0509e82f13b2caa54e92c2e0e63db89c961d09d37797d68e6 (2205 bytes)
name: "Apply issue labels to PR"

on:
  pull_request_target:
    types:
      - opened



### calcom__cal.com/d8e1b67c542615b1d03d20aaa8470703498fa00cfe795d0ba1ce0fec00293ab0 (2975 bytes)
name: E2E test
on: [push]
jobs:
  test:
    timeout-minutes: 10
    name: ${{ matrix.node }} and ${{ matrix.os }}

    env:

### calcom__cal.com/6789a2e0b5caf26d77793cd48f4

## 5) Verifiche rapide con assert
Controlli minimi per individuare inconsistenze di processo o dati corrotti.

In [12]:
assert OUTPUT_DIR.exists(), "Cartella output assente"
assert WORKFLOWS_DIR.exists(), "Cartella workflows assente"
assert len(repos_rows) > 0, "Lista repo vuota"
assert len(report_rows) > 0, "Report CSV vuoto"
assert len(workflow_files) > 0, "Nessun file trovato in output/workflows"

# Almeno una quota dei file deve sembrare YAML
yaml_like_count = sum(1 for r in clean_rows if r["inferred_is_yaml_like"])
assert yaml_like_count > 0, "Nessun file riconosciuto come YAML-like"

# Verifica dimensioni minime ragionevoli
small_files = [r for r in clean_rows if r["size_bytes"] < MIN_VALID_WORKFLOW_BYTES]
print(f"File molto piccoli (<{MIN_VALID_WORKFLOW_BYTES} bytes): {len(small_files)}")
print(f"File YAML-like stimati: {yaml_like_count}/{len(clean_rows)}")
print("Assert principali superati")

File molto piccoli (<30 bytes): 1
File YAML-like stimati: 5883/5885
Assert principali superati


## 6) Salvataggio output e artefatti
Esporta il dataset pulito in CSV e, se possibile, anche in Parquet.

In [13]:
save_info = save_dataset(clean_rows, EXPORT_CSV, EXPORT_PARQUET)
print("Export completato")
print(f"- CSV: {save_info['csv_path']}")
print(f"- Parquet: {save_info['parquet_path']}")
print(f"- Parquet salvato: {save_info['parquet_saved']}")
if save_info["parquet_error"]:
    print(f"- Parquet error: {save_info['parquet_error']}")

Export completato
- CSV: /Users/giuseppepiosorrentino/awesome-ai-agents-cicd-extract/output/dataset_workflow_files_clean.csv
- Parquet: /Users/giuseppepiosorrentino/awesome-ai-agents-cicd-extract/output/dataset_workflow_files_clean.parquet
- Parquet salvato: False


## 7) Automazione minima con cella main
Questa cella esegue l'intera pipeline in sequenza.

In [ ]:
def run_main():
    t0 = time.time()
    summary, report_rows, repos_rows = load_inputs(SUMMARY_JSON, REPORT_CSV, REPOS_CSV)
    workflow_files = list_workflow_files(WORKFLOWS_DIR)
    metrics = compute_quality_metrics(summary, report_rows, repos_rows, workflow_files)
    clean_rows = build_clean_dataset(workflow_files, report_rows)

    # Verifiche minime
    assert len(repos_rows) > 0, "Lista repo vuota"
    assert len(report_rows) > 0, "Report CSV vuoto"
    assert len(workflow_files) > 0, "Nessun workflow file trovato"

    save_info = save_dataset(clean_rows, EXPORT_CSV, EXPORT_PARQUET)

    elapsed = time.time() - t0
    print(f"Pipeline completata in {elapsed:.2f}s")
    print(f"Repo in lista: {metrics['repos_in_repo_list']}")
    print(f"Repo nel report: {metrics['repos_in_report']}")
    print(f"Repo con file reali: {metrics['repos_with_actual_files']}")
    print(f"Workflow file: {metrics['workflow_file_count']}")
    print(f"CSV esportato: {save_info['csv_path']}")
    print(f"Parquet esportato: {save_info['parquet_saved']}")

    if pd is not None:
        return pd.DataFrame(clean_rows)
    return clean_rows


RUN_MAIN = False
if RUN_MAIN:
    main_output = run_main()

## 8) Feature Engineering per Repository
Obiettivo: sintetizzare i workflow a livello repo con feature interpretabili e una descrizione testuale automatica.

In [17]:
REPO_FEATURES_CSV = OUTPUT_DIR / "dataset_repo_features.csv"
REPO_FEATURES_JSON = OUTPUT_DIR / "dataset_repo_features.json"

try:
    import yaml
except Exception:
    yaml = None


def _indent_count(line: str) -> int:
    return len(line) - len(line.lstrip(" "))


def _extract_block_children_keys(text: str, root_key: str):
    """Estrae chiavi figlie immediate di un blocco YAML root-level con parser leggero a indentazione."""
    lines = text.splitlines()
    root_idx = None
    root_indent = None

    for i, line in enumerate(lines):
        if re.match(rf"^\s*{re.escape(root_key)}\s*:\s*$", line):
            root_idx = i
            root_indent = _indent_count(line)
            break

    if root_idx is None:
        return []

    child_indent = None
    keys = []

    for line in lines[root_idx + 1:]:
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            continue

        indent = _indent_count(line)
        if indent <= root_indent:
            break

        if child_indent is None:
            child_indent = indent

        if indent == child_indent:
            m = re.match(r"^\s*([A-Za-z0-9_.-]+)\s*:\s*(?:#.*)?$", line)
            if m:
                keys.append(m.group(1))

    return keys


def _normalize_runs_on(v):
    if isinstance(v, str):
        return [v.strip()]
    if isinstance(v, list):
        out = []
        for x in v:
            if isinstance(x, str):
                out.append(x.strip())
        return out
    return []


def _deserialize_list(value):
    if not value:
        return []
    try:
        v = json.loads(value)
        if isinstance(v, list):
            return [str(x) for x in v]
    except Exception:
        pass
    return [x for x in str(value).split("|") if x]


def _serialize_list(values):
    return json.dumps(sorted(values), ensure_ascii=False)


def _features_from_yaml_obj(obj):
    feats = {
        "events": set(),
        "job_count": 0,
        "runner_labels": set(),
        "has_matrix": False,
        "has_workflow_dispatch": False,
        "has_schedule": False,
        "has_workflow_call": False,
        "uses_reusable_workflow": False,
        "actions_used": set(),
    }

    if not isinstance(obj, dict):
        return feats

    on_field = obj.get("on")
    if isinstance(on_field, str):
        feats["events"].add(on_field)
    elif isinstance(on_field, list):
        for e in on_field:
            if isinstance(e, str):
                feats["events"].add(e)
    elif isinstance(on_field, dict):
        feats["events"].update(on_field.keys())

    feats["has_workflow_dispatch"] = "workflow_dispatch" in feats["events"]
    feats["has_schedule"] = "schedule" in feats["events"]
    feats["has_workflow_call"] = "workflow_call" in feats["events"]

    jobs = obj.get("jobs")
    if isinstance(jobs, dict):
        feats["job_count"] = len(jobs)
        for _, j in jobs.items():
            if not isinstance(j, dict):
                continue

            for rr in _normalize_runs_on(j.get("runs-on")):
                feats["runner_labels"].add(rr)

            strategy = j.get("strategy")
            if isinstance(strategy, dict) and "matrix" in strategy:
                feats["has_matrix"] = True

            uses_job = j.get("uses")
            if isinstance(uses_job, str) and ".github/workflows/" in uses_job:
                feats["uses_reusable_workflow"] = True

            steps = j.get("steps")
            if isinstance(steps, list):
                for s in steps:
                    if isinstance(s, dict):
                        u = s.get("uses")
                        if isinstance(u, str):
                            feats["actions_used"].add(u)
                            if ".github/workflows/" in u:
                                feats["uses_reusable_workflow"] = True

    return feats


def features_from_workflow_text(text: str):
    base = {
        "is_yaml_like": infer_yaml_like(text),
        "events": set(),
        "job_count": 0,
        "runner_labels": set(),
        "has_matrix": False,
        "has_workflow_dispatch": False,
        "has_schedule": False,
        "has_workflow_call": False,
        "uses_reusable_workflow": False,
        "actions_used": set(),
    }

    docs = [d for d in re.split(r"(?m)^---\s*$", text) if d.strip()]

    if yaml is not None:
        for d in docs:
            try:
                obj = yaml.safe_load(d)
            except Exception:
                obj = None
            feats = _features_from_yaml_obj(obj)
            base["events"].update(feats["events"])
            base["job_count"] += feats["job_count"]
            base["runner_labels"].update(feats["runner_labels"])
            base["has_matrix"] = base["has_matrix"] or feats["has_matrix"]
            base["has_workflow_dispatch"] = base["has_workflow_dispatch"] or feats["has_workflow_dispatch"]
            base["has_schedule"] = base["has_schedule"] or feats["has_schedule"]
            base["has_workflow_call"] = base["has_workflow_call"] or feats["has_workflow_call"]
            base["uses_reusable_workflow"] = base["uses_reusable_workflow"] or feats["uses_reusable_workflow"]
            base["actions_used"].update(feats["actions_used"])
        return base

    # Fallback senza PyYAML
    base["events"].update(_extract_block_children_keys(text, "on"))
    base["job_count"] = len(_extract_block_children_keys(text, "jobs"))

    for m in re.finditer(r"(?mi)^\s*runs-on\s*:\s*(.+)$", text):
        v = m.group(1).strip().strip("[]")
        parts = [x.strip().strip("'\"") for x in v.split(",") if x.strip()]
        for p in parts:
            base["runner_labels"].add(p)

    low = text.lower()
    base["has_matrix"] = "matrix:" in low
    base["has_workflow_dispatch"] = "workflow_dispatch:" in low
    base["has_schedule"] = re.search(r"(?m)^\s*schedule\s*:\s*$", text) is not None
    base["has_workflow_call"] = "workflow_call:" in low
    base["uses_reusable_workflow"] = ".github/workflows/" in text

    for m in re.finditer(r"(?mi)^\s*uses\s*:\s*([^\s#]+)", text):
        base["actions_used"].add(m.group(1).strip())

    return base


def aggregate_repo_features(workflows_dir: Path):
    repo_acc = {}

    for p in workflows_dir.rglob("*"):
        if not p.is_file():
            continue

        rel = p.relative_to(workflows_dir)
        repo_dir = rel.parts[0] if rel.parts else ""
        repo = repo_dir.replace("__", "/") if repo_dir else ""

        if repo not in repo_acc:
            repo_acc[repo] = {
                "repo": repo,
                "repo_dir": repo_dir,
                "workflow_files": 0,
                "yaml_like_files": 0,
                "total_jobs": 0,
                "events": set(),
                "runner_labels": set(),
                "has_matrix": False,
                "has_workflow_dispatch": False,
                "has_schedule": False,
                "has_workflow_call": False,
                "uses_reusable_workflow": False,
                "actions_used": set(),
            }

        text = p.read_text(encoding="utf-8", errors="ignore")
        wf = features_from_workflow_text(text)

        acc = repo_acc[repo]
        acc["workflow_files"] += 1
        acc["yaml_like_files"] += 1 if wf["is_yaml_like"] else 0
        acc["total_jobs"] += int(wf["job_count"])
        acc["events"].update(wf["events"])
        acc["runner_labels"].update(wf["runner_labels"])
        acc["has_matrix"] = acc["has_matrix"] or wf["has_matrix"]
        acc["has_workflow_dispatch"] = acc["has_workflow_dispatch"] or wf["has_workflow_dispatch"]
        acc["has_schedule"] = acc["has_schedule"] or wf["has_schedule"]
        acc["has_workflow_call"] = acc["has_workflow_call"] or wf["has_workflow_call"]
        acc["uses_reusable_workflow"] = acc["uses_reusable_workflow"] or wf["uses_reusable_workflow"]
        acc["actions_used"].update(wf["actions_used"])

    rows = []
    for repo, acc in repo_acc.items():
        wf_count = max(acc["workflow_files"], 1)
        events_sorted = sorted(str(e) for e in acc["events"] if str(e).strip())
        runners_sorted = sorted(str(r) for r in acc["runner_labels"] if str(r).strip())
        actions_sorted = sorted(str(a) for a in acc["actions_used"] if str(a).strip())

        row = {
            "repo": repo,
            "repo_dir": acc["repo_dir"],
            "workflow_files": acc["workflow_files"],
            "yaml_like_files": acc["yaml_like_files"],
            "yaml_like_ratio": round(acc["yaml_like_files"] / wf_count, 4),
            "total_jobs": acc["total_jobs"],
            "avg_jobs_per_workflow": round(acc["total_jobs"] / wf_count, 4),
            "unique_events_count": len(events_sorted),
            "events_json": _serialize_list(events_sorted),
            "unique_runner_count": len(runners_sorted),
            "runner_labels_json": _serialize_list(runners_sorted),
            "has_matrix": acc["has_matrix"],
            "has_workflow_dispatch": acc["has_workflow_dispatch"],
            "has_schedule": acc["has_schedule"],
            "has_workflow_call": acc["has_workflow_call"],
            "uses_reusable_workflow": acc["uses_reusable_workflow"],
            "unique_actions_count": len(actions_sorted),
            "sample_actions_json": _serialize_list(actions_sorted[:10]),
        }
        rows.append(row)

    rows.sort(key=lambda x: (-x["workflow_files"], x["repo"]))
    return rows


def describe_repo(row):
    events = _deserialize_list(row.get("events_json", ""))
    runners = _deserialize_list(row.get("runner_labels_json", ""))

    ev = ", ".join(events[:4]) if events else "eventi non identificati"
    rn = ", ".join(runners[:3]) if runners else "runner non identificati"

    return (
        f"{row['repo']} ha {row['workflow_files']} workflow "
        f"({row['yaml_like_files']} YAML-like), con {row['total_jobs']} job totali "
        f"e media {row['avg_jobs_per_workflow']} job/workflow. "
        f"Trigger principali: {ev}. Runner principali: {rn}. "
        f"Matrix: {'si' if row['has_matrix'] else 'no'}, "
        f"workflow riusabili ricevuti (workflow_call): {'si' if row['has_workflow_call'] else 'no'}, "
        f"riuso di workflow esterni/interni: {'si' if row['uses_reusable_workflow'] else 'no'}."
    )


def export_repo_features(rows, out_csv: Path, out_json: Path):
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    if rows:
        keys = list(rows[0].keys()) + ["repo_description"]
        with out_csv.open("w", encoding="utf-8", newline="") as f:
            w = csv.DictWriter(f, fieldnames=keys)
            w.writeheader()
            for r in rows:
                rr = dict(r)
                rr["repo_description"] = describe_repo(r)
                w.writerow(rr)

        payload = []
        for r in rows:
            rr = dict(r)
            rr["repo_description"] = describe_repo(r)
            payload.append(rr)

        out_json.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

    return str(out_csv), str(out_json)

In [18]:
t_repo = time.time()
repo_feature_rows = aggregate_repo_features(WORKFLOWS_DIR)
repo_elapsed = time.time() - t_repo

print(f"Repo con feature estratte: {len(repo_feature_rows)}")
print(f"Tempo estrazione feature: {repo_elapsed:.2f}s")

assert len(repo_feature_rows) > 0, "Nessuna feature per repo estratta"

# Preview top 10
for r in repo_feature_rows[:10]:
    print(
        f"- {r['repo']}: workflows={r['workflow_files']}, total_jobs={r['total_jobs']}, "
        f"events={r['unique_events_count']}, runners={r['unique_runner_count']}, matrix={r['has_matrix']}"
    )

csv_path, json_path = export_repo_features(repo_feature_rows, REPO_FEATURES_CSV, REPO_FEATURES_JSON)
print("\nExport repo-level completato")
print(f"CSV: {csv_path}")
print(f"JSON: {json_path}")

print("\nDescrizioni automatiche (sample):")
for r in repo_feature_rows[:5]:
    print(f"- {describe_repo(r)}")

Repo con feature estratte: 79
Tempo estrazione feature: 1.67s
- calcom/cal.com: workflows=960, total_jobs=2832, events=12, runners=9, matrix=True
- continuedev/continue: workflows=629, total_jobs=2498, events=12, runners=8, matrix=True
- Significant-Gravitas/Auto-GPT: workflows=538, total_jobs=895, events=12, runners=6, matrix=True
- OpenDevin/OpenDevin: workflows=528, total_jobs=1561, events=11, runners=14, matrix=True
- microsoft/autogen: workflows=333, total_jobs=1450, events=10, runners=4, matrix=True
- HumanSignal/Adala: workflows=324, total_jobs=535, events=10, runners=6, matrix=True
- cpacker/MemGPT: workflows=298, total_jobs=345, events=9, runners=5, matrix=True
- camel-ai/camel: workflows=239, total_jobs=413, events=7, runners=2, matrix=True
- e2b-dev/e2b: workflows=204, total_jobs=417, events=5, runners=4, matrix=True
- eidolon-ai/eidolon: workflows=200, total_jobs=440, events=7, runners=1, matrix=True

Export repo-level completato
CSV: /Users/giuseppepiosorrentino/awesome-ai

## 9) Scoring avanzato: McCabe-like, Glue Code, Risk
Questa sezione calcola tre score per repo in scala 0-100:
- `complexity_score_mccabe_like`: complessità operativa del workflow;
- `glue_code_score`: intensità di integrazione/orchestrazione tra componenti;
- `maintainability_risk_score`: rischio manutentivo combinato.

In [19]:
REPO_SCORES_CSV = OUTPUT_DIR / "dataset_repo_scores.csv"
REPO_SCORES_JSON = OUTPUT_DIR / "dataset_repo_scores.json"


def _to_bool(v):
    if isinstance(v, bool):
        return v
    s = str(v).strip().lower()
    return s in {"1", "true", "yes", "y", "t"}


def _to_float(v, default=0.0):
    try:
        return float(v)
    except Exception:
        return default


def _safe_div(a, b):
    return (a / b) if b else 0.0


def _norm_map(values_dict):
    if not values_dict:
        return {}
    vals = list(values_dict.values())
    mn = min(vals)
    mx = max(vals)
    if mx <= mn:
        return {k: 0.0 for k in values_dict}
    return {k: (v - mn) / (mx - mn) for k, v in values_dict.items()}


def _collect_repo_aux_metrics(workflows_dir: Path):
    aux = {}

    for p in workflows_dir.rglob("*"):
        if not p.is_file():
            continue
        rel = p.relative_to(workflows_dir)
        repo_dir = rel.parts[0] if rel.parts else ""
        repo = repo_dir.replace("__", "/") if repo_dir else ""
        if not repo:
            continue

        if repo not in aux:
            aux[repo] = {
                "if_count": 0,
                "needs_count": 0,
                "uses_total": 0,
                "uses_local": 0,
                "uses_external": 0,
                "run_steps": 0,
                "external_actions_set": set(),
            }

        text = p.read_text(encoding="utf-8", errors="ignore")
        low = text.lower()

        aux[repo]["if_count"] += len(re.findall(r"(?mi)^\s*if\s*:\s*", text))
        aux[repo]["needs_count"] += len(re.findall(r"(?mi)^\s*needs\s*:\s*", text))
        aux[repo]["run_steps"] += len(re.findall(r"(?mi)^\s*run\s*:\s*", text))

        for m in re.finditer(r"(?mi)^\s*uses\s*:\s*([^\s#]+)", text):
            u = m.group(1).strip().strip("\"'")
            aux[repo]["uses_total"] += 1
            if u.startswith("./") or u.startswith(".github/"):
                aux[repo]["uses_local"] += 1
            else:
                aux[repo]["uses_external"] += 1
                aux[repo]["external_actions_set"].add(u)

    for repo, d in aux.items():
        d["unique_external_actions_count"] = len(d["external_actions_set"])
        d.pop("external_actions_set", None)

    return aux


def compute_repo_scores(repo_rows, workflows_dir: Path):
    if not repo_rows:
        return []

    aux = _collect_repo_aux_metrics(workflows_dir)

    wf = {r["repo"]: _to_float(r.get("workflow_files", 0)) for r in repo_rows}
    avg_jobs = {r["repo"]: _to_float(r.get("avg_jobs_per_workflow", 0)) for r in repo_rows}
    uniq_events = {r["repo"]: _to_float(r.get("unique_events_count", 0)) for r in repo_rows}
    uniq_runners = {r["repo"]: _to_float(r.get("unique_runner_count", 0)) for r in repo_rows}
    uniq_actions = {r["repo"]: _to_float(r.get("unique_actions_count", 0)) for r in repo_rows}

    if_per_wf = {}
    needs_per_wf = {}
    uses_per_wf = {}
    ext_ratio = {}
    local_ratio = {}
    uniq_external_actions = {}

    for r in repo_rows:
        repo = r["repo"]
        c_wf = max(_to_float(r.get("workflow_files", 0)), 1.0)
        a = aux.get(repo, {})

        if_per_wf[repo] = _safe_div(_to_float(a.get("if_count", 0)), c_wf)
        needs_per_wf[repo] = _safe_div(_to_float(a.get("needs_count", 0)), c_wf)
        uses_per_wf[repo] = _safe_div(_to_float(a.get("uses_total", 0)), c_wf)

        uses_total = _to_float(a.get("uses_total", 0))
        uses_external = _to_float(a.get("uses_external", 0))
        uses_local = _to_float(a.get("uses_local", 0))

        ext_ratio[repo] = _safe_div(uses_external, uses_total)
        local_ratio[repo] = _safe_div(uses_local, uses_total)
        uniq_external_actions[repo] = _to_float(a.get("unique_external_actions_count", 0))

    n_wf = _norm_map(wf)
    n_avg_jobs = _norm_map(avg_jobs)
    n_events = _norm_map(uniq_events)
    n_runners = _norm_map(uniq_runners)
    n_actions = _norm_map(uniq_actions)
    n_if_per_wf = _norm_map(if_per_wf)
    n_needs_per_wf = _norm_map(needs_per_wf)
    n_uses_per_wf = _norm_map(uses_per_wf)
    n_uniq_external = _norm_map(uniq_external_actions)

    out = []
    for r in repo_rows:
        repo = r["repo"]

        has_matrix = 1.0 if _to_bool(r.get("has_matrix", False)) else 0.0
        has_workflow_call = 1.0 if _to_bool(r.get("has_workflow_call", False)) else 0.0
        uses_reusable = 1.0 if _to_bool(r.get("uses_reusable_workflow", False)) else 0.0

        complexity = (
            0.30 * n_avg_jobs[repo]
            + 0.15 * n_events[repo]
            + 0.10 * n_runners[repo]
            + 0.20 * n_if_per_wf[repo]
            + 0.15 * n_needs_per_wf[repo]
            + 0.10 * has_matrix
        )

        glue = (
            0.20 * n_actions[repo]
            + 0.20 * n_uses_per_wf[repo]
            + 0.20 * ext_ratio[repo]
            + 0.10 * local_ratio[repo]
            + 0.15 * n_uniq_external[repo]
            + 0.10 * uses_reusable
            + 0.05 * has_workflow_call
        )

        risk = (
            0.45 * complexity
            + 0.35 * glue
            + 0.20 * n_wf[repo]
        )

        rr = dict(r)
        rr["if_per_workflow"] = round(if_per_wf[repo], 4)
        rr["needs_per_workflow"] = round(needs_per_wf[repo], 4)
        rr["uses_per_workflow"] = round(uses_per_wf[repo], 4)
        rr["external_action_ratio"] = round(ext_ratio[repo], 4)
        rr["local_action_ratio"] = round(local_ratio[repo], 4)
        rr["unique_external_actions_count"] = int(uniq_external_actions[repo])
        rr["complexity_score_mccabe_like"] = round(complexity * 100, 2)
        rr["glue_code_score"] = round(glue * 100, 2)
        rr["maintainability_risk_score"] = round(risk * 100, 2)

        out.append(rr)

    out.sort(key=lambda x: (-x["maintainability_risk_score"], x["repo"]))
    return out


def export_repo_scores(rows, out_csv: Path, out_json: Path):
    if not rows:
        return str(out_csv), str(out_json)

    keys = list(rows[0].keys())
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    with out_csv.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        w.writerows(rows)

    out_json.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
    return str(out_csv), str(out_json)

In [20]:
repo_scores_rows = compute_repo_scores(repo_feature_rows, WORKFLOWS_DIR)
score_csv, score_json = export_repo_scores(repo_scores_rows, REPO_SCORES_CSV, REPO_SCORES_JSON)

print(f"Repo score rows: {len(repo_scores_rows)}")
print(f"CSV: {score_csv}")
print(f"JSON: {score_json}")

print("\nTop 10 per maintainability_risk_score:")
for r in repo_scores_rows[:10]:
    print(
        f"- {r['repo']}: risk={r['maintainability_risk_score']}, "
        f"complexity={r['complexity_score_mccabe_like']}, glue={r['glue_code_score']}"
    )

print("\nBottom 10 per maintainability_risk_score:")
for r in repo_scores_rows[-10:]:
    print(
        f"- {r['repo']}: risk={r['maintainability_risk_score']}, "
        f"complexity={r['complexity_score_mccabe_like']}, glue={r['glue_code_score']}"
    )

Repo score rows: 79
CSV: /Users/giuseppepiosorrentino/awesome-ai-agents-cicd-extract/output/dataset_repo_scores.csv
JSON: /Users/giuseppepiosorrentino/awesome-ai-agents-cicd-extract/output/dataset_repo_scores.json

Top 10 per maintainability_risk_score:
- continuedev/continue: risk=71.42, complexity=71.81, glue=74.32
- calcom/cal.com: risk=65.07, complexity=58.53, glue=53.51
- OpenDevin/OpenDevin: risk=59.96, complexity=60.64, glue=61.94
- Significant-Gravitas/Auto-GPT: risk=48.52, complexity=39.08, glue=56.38
- HumanSignal/Adala: risk=47.76, complexity=35.47, glue=71.61
- microsoft/autogen: risk=46.53, complexity=52.56, glue=45.58
- airtai/fastagency: risk=42.63, complexity=61.03, glue=39.4
- homanp/superagent: risk=38.65, complexity=53.93, glue=39.04
- e2b-dev/e2b: risk=36.55, complexity=34.61, glue=47.83
- zabirauf/AutoGPT.js: risk=35.12, complexity=44.47, glue=43.17

Bottom 10 per maintainability_risk_score:
- proficientai/js: risk=8.85, complexity=2.58, glue=21.96
- metauto-ai/GPT